In [1]:
import gensim.downloader as api
from gensim.utils import simple_preprocess
import requests
import numpy as np
import matplotlib.pyplot as plt
glove = api.load("glove-wiki-gigaword-100")
url_part1 = "https://huggingface.co/datasets/usaaio-official/"
url_part2 = "2026_USAAIO_Round1_public/raw/main/"
url_part3 = "2026_USAAIO_Round1_NLP.txt"
url = url_part1 + url_part2 + url_part3
text = requests.get(url).text
tokens = simple_preprocess(text)
tokens = [token for token in tokens if token in glove]
print(len(tokens))
tokens = list(set(tokens))
print(len(tokens))
vectors = [glove[word] for word in tokens]

[==================================================] 100.0% 128.1/128.1MB downloaded
225
158


In [2]:
print(len(tokens))

158


In [8]:
vectors_arr = np.stack(vectors)
vectors_arr.shape

(158, 100)

In [35]:
norms = np.sqrt(np.sum(vectors_arr**2, axis=1, keepdims=True))
W = vectors_arr/norms
np.sum([x**2 for x in W[0]])

np.float32(1.0)

In [37]:
S = W @ W.T
S.shape

(158, 158)

In [57]:
for i in range(len(tokens)):
    S[i, i] = 0

In [59]:
sum_dict = {tokens[i]: tokens[np.argmax(S[i], )] for i in range(len(tokens))}
sum_dict['bright']

'lights'

In [91]:
u, s, v = np.linalg.svd(W)
m, n = W.shape
sigma = np.zeros((m, n))
np.fill_diagonal(sigma, s)
u @ sigma @ v

array([[ 0.02465622,  0.03788213,  0.07484189, ..., -0.00490057,
         0.24730963,  0.1805606 ],
       [-0.03790732, -0.08980559, -0.01316939, ..., -0.01664126,
         0.05705041,  0.01008618],
       [ 0.02598584, -0.05070248,  0.01739415, ..., -0.01709239,
         0.18117451,  0.08140115],
       ...,
       [-0.0602464 ,  0.01815618, -0.13313773, ...,  0.04114566,
         0.16662131, -0.02563944],
       [-0.02324469,  0.06590082,  0.05781288, ...,  0.05686654,
         0.11569474,  0.05705236],
       [-0.03523207,  0.01869297,  0.0621176 , ..., -0.0047553 ,
        -0.11789052,  0.10154829]], shape=(158, 100))

In [104]:
u @ sigma @ v @ v.T @ sigma.T @ u.T

array([[0.99999991, 0.25272449, 0.57467401, ..., 0.17630219, 0.34891444,
        0.27106719],
       [0.25272449, 0.99999988, 0.51479417, ..., 0.25884896, 0.45193116,
        0.39774208],
       [0.57467401, 0.51479417, 0.99999987, ..., 0.39176154, 0.50222278,
        0.4318585 ],
       ...,
       [0.17630219, 0.25884896, 0.39176154, ..., 0.99999995, 0.31372344,
        0.18184326],
       [0.34891444, 0.45193116, 0.50222278, ..., 0.31372344, 0.99999992,
        0.49204647],
       [0.27106719, 0.39774208, 0.4318585 , ..., 0.18184326, 0.49204647,
        0.9999999 ]], shape=(158, 158))

In [110]:
Q = u
lambd = sigma @ sigma.T
Q @ lambd @ Q.T

array([[0.99999991, 0.25272449, 0.574674  , ..., 0.17630219, 0.34891444,
        0.27106719],
       [0.25272449, 0.99999988, 0.51479417, ..., 0.25884897, 0.45193115,
        0.39774208],
       [0.574674  , 0.51479417, 0.99999986, ..., 0.39176154, 0.50222278,
        0.4318585 ],
       ...,
       [0.17630219, 0.25884897, 0.39176154, ..., 0.99999995, 0.31372344,
        0.18184326],
       [0.34891444, 0.45193115, 0.50222278, ..., 0.31372344, 0.99999992,
        0.49204647],
       [0.27106719, 0.39774208, 0.4318585 , ..., 0.18184326, 0.49204647,
        0.9999999 ]], shape=(158, 158))